# **Import Libraries**

In [1]:
pip install torch torchvision torchaudio

INFO: pip is looking at multiple versions of torch to determine which version is compatible with other requirements. This could take a while.
^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import os

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# **Data Preprossing**

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self, base_path, transform=None):
        self.dir = base_path
        self.classes = os.listdir(self.dir)
        self.mapping = {class_name: idx for idx, class_name in enumerate(self.classes)}      
        self.images = []
        self.labels = []
        self.transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor()
        ])
        for clas in self.classes:
            class_path = os.path.join(self.dir, clas)
            for image_name in os.listdir(class_path):
                image_path = os.path.join(class_path, image_name)
                if os.path.isfile(image_path):
                    self.images.append(image_path)
                    self.labels.append(self.mapping[clas])
                            
    def __len__(self):
        return len(self.images)
        
    def __getitem__(self, idx):
        image_path = self.images[idx]
        label = self.labels[idx]
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image, label

In [ ]:
class CustomTestDataset(Dataset):
    def __init__(self, base_path, dataframe, transform=None):
        self.dir = base_path
        self.transform = transform
        self.dataframe = dataframe
        self.image_files = dataframe['filename'].tolist()

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        filename = self.image_files[idx]
        image_path = os.path.join(self.dir, filename)
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        return image, filename

## **Model Initilization**

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = self.relu(out)
        return out

In [ ]:
class CustomModel(nn.Module):
    def __init__(self, num_classes=9):
        super(CustomModel, self).__init__()
        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        
        self.layer1 = self.layers(64, 2, stride=1)
        self.layer2 = self.layers(128, 2, stride=2)
        self.layer3 = self.layers(256, 2, stride=2)
        self.layer4 = self.layers(512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def layers(self, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(ResidualBlock(self.in_channels, out_channels, stride=s))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        out = F.softmax(x, dim = 1)
        return out 

    def train_model(self, train_loader, criterion, optimizer, num_epochs, device):
        self.train()
        for epoch in range(num_epochs):
            curr_loss = 0.0
            for i, (images, labels) in enumerate(train_loader):
                images = images.to(device)
                labels = labels.to(device)
                outputs = self(images)
                loss = criterion(outputs, labels)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                curr_loss += loss.item()
            epoch_loss = curr_loss / len(train_loader)
            print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

    def predict(self, test_path, submission_csv):
        self.eval()
        submission_df = pd.read_csv(submission_csv)
        test_transform = T.Compose([
            T.Resize((224, 224)),
            T.ToTensor()
        ])
        submission_dataset = CustomTestDataset(base_path=test_path, dataframe=submission_df, transform=test_transform)
        submission_loader = DataLoader(submission_dataset, batch_size=64, shuffle=False)
        all_predictions = []
        with torch.no_grad():
            for images, index in submission_loader:
                images = images.to(device)
                lab = self(images)
                output = torch.argmax(lab, dim=1)
                all_predictions.extend([p.item() for p in output])
        final_labels = [p + 1 for p in all_predictions]
        submission_df['label'] = final_labels
        submission_df.to_csv("submission.csv", index=False)

# **Submission and Predictions**

In [ ]:
train_path = "/kaggle/input/AI-OF-GOD-4/aog_data/train"
test_path = "/kaggle/input/AI-OF-GOD-4/aog_data/test/images"
submission_path = "/kaggle/input/AI-OF-GOD-4/aog_data/sample_submission.csv"

epochs = 30
learning_rate = 0.001

train_dataset = CustomImageDataset(base_path=train_path)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomModel(num_classes=len(train_dataset.classes))
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

model.train_model(train_loader, criterion, optimizer, epochs, device)


In [ ]:
model.predict(test_path=test_path, submission_csv=submission_path)